# Resolução 03 — IA aplicada: classificação de risco de fogo

**Curso:** Tecnologia, Dados e IA na Gestão Ambiental
**Encontros:** 6 e 9

Este notebook resolvedor treina uma **árvore de decisão** para prever risco de fogo a partir
de variáveis climáticas. Ele roda do início ao fim com **Runtime → Run all** e baixa os dados
automaticamente.

O foco não é "acertar mais": é **entender o erro**, comparar com uma baseline e preencher uma
ficha de modelo auditável.

> Os dados são sintéticos e didáticos. Não representam o município nem servem para alerta real.

## Passo 0 — Ambiente e dados

O Colab já traz pandas, Matplotlib e scikit-learn. A célula instala o scikit-learn apenas se
estiver faltando.

In [ ]:
from pathlib import Path
import urllib.request

try:
    import sklearn
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])
    import sklearn

import pandas as pd
import matplotlib.pyplot as plt

BASE_DADOS = "https://raw.githubusercontent.com/Marquezbertin/Curso-Tecnologia-Dados-Gestao-Ambiental/main/06-DADOS/exemplos"

def baixar(nome, base=BASE_DADOS):
    destino = Path(nome)
    if not destino.exists():
        print(f"Baixando {nome}...")
        urllib.request.urlretrieve(f"{base}/{nome}", destino)
    print(f"Arquivo pronto: {destino.resolve()}")
    return destino

ARQUIVO = baixar("exemplo-risco-fogo-sintetico.csv")
print("scikit-learn:", sklearn.__version__)

## Passo 1 — Ler os dados e olhar o balanceamento

Antes de treinar, verifique quantas linhas existem e qual é a proporção de cada classe. Um
modelo pode parecer bom apenas por chutar sempre a classe majoritária.

In [ ]:
df = pd.read_csv(ARQUIVO)

print("Linhas:", len(df), "| Colunas:", df.shape[1])
print("\nContagem de risco_fogo:")
print(df["risco_fogo"].value_counts().to_string())
print("\nProporção de risco=1:", round(df["risco_fogo"].mean(), 3))
df.head()

In [ ]:
df.groupby("risco_fogo")[["temp_max_c", "umidade_rel_pct", "vento_kmh", "chuva_mm"]].mean().round(2)

### Leitura

- 300 registros, sendo **52 com risco (17,3%)** e **248 sem risco (82,7%)**.
- Em média, dias com risco têm temperatura máxima maior (32,75 °C contra 28,77 °C), umidade
  menor (46,48% contra 54,17%) e menos chuva (2,14 mm contra 4,55 mm).
- A média é só um começo: a distribuição se sobrepõe. É por isso que o modelo erra.

## Passo 2 — Separar treino e teste

Nunca avalie o modelo com os mesmos dados que ele usou para aprender. Usamos 70% para treino
e 30% para teste, mantendo a proporção das classes com `stratify`.

In [ ]:
from sklearn.model_selection import train_test_split

entradas = ["temp_max_c", "umidade_rel_pct", "vento_kmh", "chuva_mm"]
X = df[entradas]
y = df["risco_fogo"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Treino:", X_treino.shape, "| Teste:", X_teste.shape)
print("Risco no treino:", y_treino.value_counts().to_dict())
print("Risco no teste:", y_teste.value_counts().to_dict())

## Passo 3 — Baseline: o "chute inteligente"

A baseline mais frequente prevê sempre a classe majoritária. Se o modelo não superar isso de
forma útil, ele não aprendeu nada relevante.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_treino, y_treino)

acc_baseline = accuracy_score(y_teste, baseline.predict(X_teste))
print("Acurácia da baseline (chutar sempre 'sem risco'):", round(acc_baseline, 3))

**Resposta:** a baseline acerta **0,822** (82,2%) sem detectar nenhum dia de risco. Isso mostra
por que **acurácia sozinha engana** em dados desbalanceados.

## Passo 4 — Treinar a árvore de decisão

`max_depth=3` limita a árvore para evitar decorar o treino. `random_state=42` garante
reprodutibilidade.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo.fit(X_treino, y_treino)
print("Modelo treinado.")

## Passo 5 — Avaliar com matriz de confusão

A matriz mostra quatro resultados: verdadeiros negativos, falsos positivos, falsos negativos
e verdadeiros positivos.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

y_pred = modelo.predict(X_teste)

print("Acurácia:", round(accuracy_score(y_teste, y_pred), 3))
print("\nMatriz de confusão:")
print(confusion_matrix(y_teste, y_pred))
print("\nRelatório de classificação:")
print(classification_report(y_teste, y_pred, digits=3))

ConfusionMatrixDisplay.from_predictions(
    y_teste,
    y_pred,
    display_labels=["sem risco", "risco"],
    cmap="Greens",
)
plt.title("Matriz de confusão — árvore com max_depth=3")
plt.show()

### Resposta da matriz

| | Previsto sem risco | Previsto com risco |
|---|---|---|
| **Real sem risco** | 72 (VN) | 2 (FP) |
| **Real com risco** | 8 (FN) | 8 (VP) |

- **Acurácia:** 0,889 (88,9%).
- **Recall da classe risco:** 8 / (8 + 8) = **0,500**. O modelo deixa passar **metade** dos
  dias de risco.
- **Precisão da classe risco:** 8 / (8 + 2) = **0,800**. Quando ele alerta, acerta 80%.
- **Falsos negativos:** 8 dias de risco classificados como "sem risco".
- **Falsos positivos:** 2 dias sem risco classificados como "risco".

Na gestão ambiental, o **falso negativo** costuma ser o erro mais grave: o alerta não chega.
O custo do falso positivo é mobilizar equipe sem necessidade; o custo do falso negativo pode
ser perda ambiental e risco à comunidade. A escolha do limiar é uma decisão política e
técnica, não só do modelo.

## Passo 6 — Ver as regras aprendidas

A árvore é interpretável: cada caminho é uma regra do tipo "se chuva ≤ X e umidade ≤ Y...".

In [ ]:
from sklearn.tree import export_text

print(export_text(modelo, feature_names=entradas))

### Leitura das regras

O primeiro corte é `chuva_mm <= 1.05`. Dentro desse grupo, a árvore separa por temperatura e
umidade. Um caminho de risco é:

- chuva ≤ 1,05 mm;
- temperatura máxima > 29,75 °C;
- umidade relativa ≤ 44,25%.

Isso é coerente com a ideia de tempo seco e quente, mas é uma regra aprendida de **dados
sintéticos**. Não a trate como lei ecológica.

## Passo 7 — Importância das variáveis

In [ ]:
importancia = pd.Series(modelo.feature_importances_, index=entradas).sort_values()

importancia.plot(
    kind="barh",
    figsize=(7, 3),
    color="#16724a",
    title="Importância das variáveis na árvore",
)
plt.xlabel("Importância")
plt.tight_layout()
plt.show()

print(importancia.round(3).to_string())

**Resposta:** temperatura máxima (0,373), umidade (0,361) e chuva (0,241) concentram quase
toda a decisão; vento contribui pouco (0,024). Importância não é causalidade: ela descreve o
que **este modelo** usou, não o que causa o fogo.

## Passo 8 — Exercício: variar a profundidade

A profundidade controla o trade-off entre underfitting (modelo simples demais) e overfitting
(modelo que decora o treino). A tabela calcula acurácia, precisão, recall e F1 para cada
profundidade.

In [ ]:
resultados = []

for profundidade in range(1, 11):
    m = DecisionTreeClassifier(max_depth=profundidade, random_state=42)
    m.fit(X_treino, y_treino)
    p = m.predict(X_teste)

    tn, fp, fn, tp = confusion_matrix(y_teste, p).ravel()

    resultados.append({
        "max_depth": profundidade,
        "acuracia": accuracy_score(y_teste, p),
        "precisao": tp / (tp + fp) if (tp + fp) else 0,
        "recall": tp / (tp + fn) if (tp + fn) else 0,
        "f1": 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0,
    })

tabela = pd.DataFrame(resultados).round(3)
tabela

### Resposta

| max_depth | Acurácia | Precisão | Recall | F1 |
|---:|---:|---:|---:|---:|
| 1 | 0,822 | 0,000 | 0,000 | 0,000 |
| 2 | 0,822 | 0,500 | **0,562** | 0,529 |
| 3 | **0,889** | **0,800** | 0,500 | **0,615** |
| 4 | 0,856 | 0,615 | 0,500 | 0,552 |
| 5 | 0,844 | 0,562 | 0,562 | 0,562 |
| 6–10 | 0,833–0,844 | 0,533–0,571 | 0,500–0,562 | 0,516–0,533 |

- Profundidade **1** é underfitting: não prevê nenhum risco.
- Profundidade **2** tem o melhor **recall** (0,562), mas precisão baixa (0,500).
- Profundidade **3** tem a melhor **acurácia** (0,889) e melhor F1 (0,615), mas ainda perde
  metade dos dias de risco.
- A partir de 5, o desempenho não melhora de forma consistente. Com 300 linhas, diferenças
  pequenas podem ser ruído.

**Conclusão:** não escolha a profundidade só pela acurácia. Defina antes qual erro é mais
custoso no seu problema.

## Passo 9 — Exercício: peso das classes

Uma forma de aumentar o recall é dar mais peso à classe minoritária. Isso reduz falsos
negativos, mas aumenta falsos positivos. A célula compara.

In [ ]:
comparacao_pesos = []

for peso in [None, "balanced"]:
    for profundidade in [2, 3, 4]:
        m = DecisionTreeClassifier(
            max_depth=profundidade,
            random_state=42,
            class_weight=peso,
        )
        m.fit(X_treino, y_treino)
        p = m.predict(X_teste)
        tn, fp, fn, tp = confusion_matrix(y_teste, p).ravel()

        comparacao_pesos.append({
            "class_weight": "sem peso" if peso is None else "balanced",
            "max_depth": profundidade,
            "acuracia": accuracy_score(y_teste, p),
            "precisao": tp / (tp + fp) if (tp + fp) else 0,
            "recall": tp / (tp + fn) if (tp + fn) else 0,
            "falsos_negativos": fn,
            "falsos_positivos": fp,
        })

pd.DataFrame(comparacao_pesos).round(3)

### Resposta

- Com `class_weight="balanced"` e profundidade 2, o recall sobe de **0,562** para **0,750**,
  mas a acurácia cai de **0,822** para **0,678** e os falsos positivos sobem para 25.
- O modelo passa a "desconfiar" mais: alerta mais vezes, inclusive em dias sem risco.
- **Não existe almoço grátis:** aumentar recall custa precisão. A pergunta é: no seu contexto,
  qual erro é pior? Para alerta de queimada, o falso negativo tende a ser mais grave; para
  mobilização de equipe cara, o falso positivo pesa.

**Resposta de estudante aceitável:** escolher `balanced` se o objetivo for não perder dias de
risco, desde que haja verificação humana antes de qualquer ação.

## Passo 10 — Validação cruzada

Uma única divisão treino/teste pode ter sorte ou azar. A validação cruzada repete a avaliação
em 5 partições.

In [ ]:
from sklearn.model_selection import cross_validate

cv = cross_validate(
    DecisionTreeClassifier(max_depth=3, random_state=42),
    X,
    y,
    cv=5,
    scoring=["accuracy", "recall", "precision"],
)

resumo_cv = pd.DataFrame(cv).drop(columns=["fit_time", "score_time"])
print(resumo_cv.round(3).to_string(index=False))
print("\nMédia:")
print(resumo_cv.mean().round(3).to_string())

### Resposta

A acurácia média fica em torno de **0,883**, mas o recall médio é cerca de **0,400** e varia
muito entre as partições (0,200 a 0,636). Traduzindo: em alguns recortes o modelo perde a
maioria dos dias de risco. **A métrica única esconde instabilidade.**

Com apenas 300 registros e 17% de positivos, a variância é esperada. O caminho é coletar mais
dados, usar validação temporal e testar em outra região.

## Ficha de modelo preenchida

| Campo | Resposta |
|---|---|
| Nome do modelo | Árvore de decisão para risco de fogo (didática) |
| O que ele faz | Classifica risco binário (0/1) a partir de 4 variáveis climáticas |
| Dados de treino | Exemplo sintético do repositório, 300 registros, sem período real |
| Variáveis de entrada | temperatura máxima, umidade relativa, vento, chuva |
| Rótulo previsto | `risco_fogo` (0 = sem risco; 1 = risco) |
| Métrica e resultado | Acurácia 0,889; recall da classe risco 0,500; precisão 0,800 |
| Falsos positivos / negativos | 2 falsos positivos / 8 falsos negativos (teste de 90 linhas) |
| Onde ele erra mais | Em dias de risco com temperatura moderada e umidade alta; perde metade dos positivos |
| Quem é afetado pelo erro | Comunidades em área de risco que deixam de receber alerta; equipes mal mobilizadas |
| Limitação principal | Dados sintéticos, pequenos, de uma só região, sem variáveis sazonais e temporais |
| Uso recomendado | Ensino, discussão de métricas e triagem com revisão humana |
| Uso não recomendado | Alerta oficial, laudo, fiscalização automática ou decisão sem humano |

### O que a ficha ainda precisa

- data de treinamento e versão dos dados;
- código e parâmetros (`max_depth=3`, `random_state=42`);
- distribuição das classes no treino e no teste;
- análise dos erros por subgrupo (região, estação do ano);
- responsável técnico e canal de contestação.

## Discussão ética com respostas

1. **Se o modelo errar um falso negativo, quem pode ser prejudicado?**
   A comunidade na área de risco, que não recebe alerta; brigadistas e defesa civil, que
   chegam tarde; o ecossistema, que perde área e biodiversidade.

2. **Se os dados de treino vierem só de uma região, o modelo funciona em outra?**
   Não automaticamente. Clima, vegetação, solo e práticas de uso do fogo mudam. Isso é
   *distribution shift*. É preciso validar com dados locais antes de usar.

3. **Quem decide o que é "risco"? O modelo ou a gestão pública?**
   A gestão pública define o critério e assume a responsabilidade. O modelo estima
   probabilidade; ele não decide o que é aceitável para a sociedade.

4. **O que precisa estar escrito na ficha para o modelo ser auditável?**
   Fonte e versão dos dados, variáveis, rótulo, métricas, matriz de confusão, análise de
   erros, limitações, uso pretendido, responsável e data. Sem isso, não há auditoria.

## Como melhorar este modelo (respostas)

- Coletar mais dados e incluir série temporal (dias anteriores, estação do ano).
- Validar em outra região e em outro ano (validação temporal).
- Tratar o desbalanceamento com peso, reamostragem ou limiar ajustado.
- Testar modelos mais robustos (random forest, gradient boosting) e comparar com a baseline.
- Fazer análise de erro por subgrupo e revisão humana.
- Publicar ficha de modelo e canal de contestação.

## Checklist de entrega

- [ ] O notebook rodou inteiro com **Run all**.
- [ ] A matriz de confusão foi interpretada, não só copiada.
- [ ] A acurácia foi comparada com a baseline.
- [ ] O recall e a precisão foram explicados no contexto.
- [ ] A ficha de modelo está preenchida.
- [ ] A discussão ética foi respondida.
- [ ] O texto não trata o modelo como oráculo.

**Próximo notebook resolvido:** [`04-resolucao-visao-computacional-satelite.ipynb`](04-resolucao-visao-computacional-satelite.ipynb)